In [17]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import norm
import pandas as pd
from scipy import stats
from itertools import product
from scipy import linalg
from functools import partial

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.dgp import GaussianNetwork, BernoulliNetwork
from src.metrics import (
    TrueRejection,
    FalseRejection,
    Rejection,
    RelativeFrobeniusNorm,
    ComputeAll,
    ProcrustesDistance
)

from src.helper_functions._metrics_helper import rv_coefficient, rv_coefficient_adjusted, cvm_stat_multivariate

from src.methods import (
    PermutationTest,
    RVPermutationTest,
    FitIndependent,
    LLKRatioTest,
    DiffusionCorrelation,
    CanonicalCorrelationTest,
    QAP
)
from src.solvers.binary_network import MLE_logistic
from src.solvers.MaMa_uuuuu import pgd_fit_wrapper, pgd_fit
from src.solvers.weighted_network import MLE_gaussian, ASE
from src.helper_functions.simulation_functions import run_simulation
from src.helper_functions.analyse_functions import aggregate_results
from src.helper_functions.plot_functions import plot_grid, plot_with_bands
import scipy.stats as stats
from src.helper_functions.plot_functions import visualise_latent
from functools import partial

In [ ]:
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# TODO:
# - this could be sped up by having dpg run once and then feed data to each arg combination
# (it has a specific name i don't remember not)
# - add intermediate save

def run_scenario(metrics, args, seed, method_params=None):
    """Run a single scenario of the simulation.

    Parameters
    ----------
    metrics : list of BaseMetric
        list of metrics to compute
    args : dict
        arguments for the simulation scenario. Should contain 'setup' key with
        (dgp, method) tuple.

    Returns
    -------
    dict
        Dictionary containing the computed metrics.
    """
    rng = np.random.default_rng(seed)
    args['rng'] = rng
    
    if args.get("data") is None:
        dgp, solver = args["setup"]
        dgp = dgp(**args)
        data = dgp.generate()
        args['dgp_name'] = dgp.get_name()
    else:
        data = args["data"]
        solver = None
    
    method = args["method"]
    method = method(solver=solver, **args)
    
    args['method_name'] = method.get_name()
    args['dgp_name'] = data['dgp_name']

    method.fit(data, **(method_params if method_params else {}))
    results = method.get_estimated()

    out_metrics = {metric.get_name(): metric(results) for metric in metrics}

    out_metrics["args"] = args
    return out_metrics

In [19]:
def run_scenario_wrapper(args):
    """Wrapper to unpack args for pool.map"""
    args, metrics, method_params, seed = args
    return run_scenario(metrics, args, seed=seed, method_params=method_params)

In [20]:
def run_simulation_parallel(
    nsim, factorial_design, metrics, method_params=None, rng=None, n_jobs=None
):
    if rng is None:
        rng = np.random.default_rng()

    if not isinstance(factorial_design, list):
        raise ValueError("factorial_design must be a list of tuples")

    if n_jobs is None:
        n_jobs = cpu_count()
        
    # Create all scenario arguments upfront (flattened structure)
    all_scenarios = [
        (args, metrics, method_params) for i in range(nsim) for args in factorial_design
    ]

    
    total_scenarios = len(all_scenarios)
    
    child_seeds = rng.spawn(total_scenarios)
    
    all_scenarios_seed = [
        (*scenario, seed) for scenario, seed in zip(all_scenarios, child_seeds)
    ]
    
    # Shuffle scenarios for better parallelisation
    rng.shuffle(all_scenarios_seed)

    # Better chunk size: balance between overhead and load distribution
    chunk_size = max(1, total_scenarios // (n_jobs * 32))

    results = []
    with Pool(processes=n_jobs) as pool:
        with tqdm(total=total_scenarios, desc="Running scenarios") as pbar:
            # Use imap_unordered for better performance (order doesn't matter)
            for result in pool.imap_unordered(
                run_scenario_wrapper, all_scenarios_seed, chunksize=chunk_size
            ):
                results.append(result)
                pbar.update(1)

    return results

In [21]:
def run_simulation(
    nsim,
    factorial_design,
    metrics,
    method_params=None,
    parallel=False,
    rng=None,
    n_jobs=None,
    data=None
):
    """Run a simulation study.

    Parameters
    ----------
    nsim : _type_
        _description_
    factorial_design : _type_
        _description_
    metrics : _type_
        _description_
    method_params : _type_, optional
        _description_, by default None
    parallel : bool, optional
        _description_, by default False
    rng : _type_, optional
        _description_, by default None
    n_jobs : _type_, optional
        _description_, by default None
    data : dict, optional
       Dictionary containing keys 'estimate_latent_x', 'estimate_latent_y', 
       'true_latent_x', and 'true_latent_y'.

    Returns
    -------
    _type_
        _description_
    """
    if parallel:
        return run_simulation_parallel(
            nsim=nsim,
            factorial_design=factorial_design,
            metrics=metrics,
            method_params=method_params,
            rng=rng,
            n_jobs=n_jobs,
        )

    if rng is None:
        rng = np.random.default_rng()

    results = []
    for i in range(nsim):
        print(f"Simulation {i + 1} of {nsim}")
        for args in tqdm(factorial_design, desc="Running scenarios"):
            scenario_out = run_scenario(metrics, args, method_params=method_params)
            results.append(scenario_out)

    return results

In [6]:
import h5py

def load_hdf5(path):
    def read_obj(obj):
        # dataset -> return array directly
        if isinstance(obj, h5py.Dataset):
            data = obj[()]
            if len(obj.attrs) == 0:
                return data
            else:
                return {"_data": data, "_attrs": dict(obj.attrs)}

        # group -> recurse
        if isinstance(obj, h5py.Group):
            out = {}

            if len(obj.attrs) > 0:
                out["_attrs"] = dict(obj.attrs)

            for key, item in obj.items():
                out[key] = read_obj(item)

            return out

    with h5py.File(path, "r") as f:
        return {k: read_obj(v) for k, v in f.items()}

data = load_hdf5('data.h5')

In [7]:
nsim = 5
n = [50, 100, 200]
k = [3]
rho = [0]
alpha = [0.05]
marginals = ['chi 5']
edge_var = [1]

method = [
    partial(RVPermutationTest, permutation_type="latent"),
    partial(RVPermutationTest, permutation_type="observed"),
    LLKRatioTest,
    QAP,
    DiffusionCorrelation,
    partial(CanonicalCorrelationTest, permutation_type="latent"),
    partial(CanonicalCorrelationTest, permutation_type="observed"),
]

npermutations = [100]
metrics = [ComputeAll()]
approximation = ["F-distr"]

solver = partial(pgd_fit_wrapper, num_iters=500, backend='jax')

model = partial(BernoulliNetwork, center_latent=True, self_loops=False)

setup = [
    (model, solver),
]

rng = np.random.default_rng(1)    

param_names = [
    "setup",
    "method",
    "n",
    "k",
    "rho",
    "alpha",
    "marginals",
    "edge_var",
    "approximation",
    "npermutations"
]

param_values = product(
    setup, method, n, k, rho, alpha, marginals, edge_var, approximation, npermutations
)

factorial_design = [dict(zip(param_names, v)) for v in param_values]

for (i, x) in enumerate(factorial_design):
    x['data'] = data[f'iteration_{i}']

In [22]:
out = run_simulation(
    nsim=nsim,
    metrics=metrics,
    factorial_design=factorial_design,
    rng=rng,
    parallel=True,
)

out = pd.DataFrame(out)

Running scenarios:   0%|          | 0/105 [00:00<?, ?it/s]Process SpawnPoolWorker-116:
Process SpawnPoolWorker-119:
Traceback (most recent call last):
Traceback (most recent call last):
Process SpawnPoolWorker-118:
Traceback (most recent call last):
Process SpawnPoolWorker-117:
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'run_scenari

KeyboardInterrupt: 